# 05 Resilience Under Execution Failures (Agent-CLIs, 2026)

## What This Lesson Is
Build bounded retry/fallback execution for unstable agent CLI operations.

## Scientific Lens
- Concept: Operational resilience in agent execution pipelines
- Measure: Completion rate under injected command failures
- Validity Limit: CLI failure injection in notebooks may not match CI/CD orchestration failure modes.


## How It Works
1. Simulate deterministic failure sequence.
2. Apply retry policy.
3. Attempt live CLI execution with bounded retry.


In [ ]:
print("Resilience lesson preflight complete")


## Code Walkthrough
The next two code cells are intentionally split:
- `Deterministic Demo`: always runnable and concept-focused.
- `Live Demo`: executes a real provider/CLI flow with explicit graceful-skip behavior.


In [ ]:
# Deterministic Demo
outcomes = iter([1, 1, 0])  # fail, fail, success
attempts = 0
max_attempts = 4

while attempts < max_attempts:
    attempts += 1
    rc = next(outcomes, 1)
    print("attempt", attempts, "rc", rc)
    if rc == 0:
        break
else:
    raise RuntimeError("all attempts failed")

assert attempts == 3


In [ ]:
# Live Demo
import shutil
import subprocess
import time

cmds = [["codex", "exec", "In one sentence, define graceful degradation."], ["claude", "-p", "In one sentence, define graceful degradation."], ["opencode", "run", "In one sentence, define graceful degradation."]]
selected = None
for c in cmds:
    if shutil.which(c[0]) is not None:
        selected = c
        break

if selected is None:
    print("Skipping live resilience run: no compatible non-interactive CLI available.")
else:
    for attempt in range(1, 4):
        p = subprocess.run(selected, capture_output=True, text=True, timeout=60)
        if p.returncode == 0:
            print((p.stdout or p.stderr).strip()[:1000])
            break
        print("attempt failed:", attempt, p.returncode)
        time.sleep(0.3 * attempt)
    else:
        print("retry budget exhausted")


## Applied Labs
1. Implement per-error retry policy (retry timeouts, fail fast on auth).
2. Add fallback from one CLI to another and measure completion gain.
3. Track retry-related latency overhead and define acceptable bounds.

## Validation Checklist
- Retry logic is bounded and observable.
- Failure reasons are logged with enough context.
- Fallback/skip path avoids hanging execution.

## Further Reading
- [AWS Retry Best Practices](https://aws.amazon.com/builders-library/timeouts-retries-and-backoff-with-jitter/)
- [Google SRE Cascading Failures](https://sre.google/sre-book/addressing-cascading-failures/)
- [Process Resilience Patterns](https://martinfowler.com/bliki/CircuitBreaker.html)
